# LAB 08 — Pretrained Transformers and BERT Fine-Tuning


In [1]:
import pandas as pd
from IPython.display import display

## 9. Calculation Exercise 1 — MLM Loss
Đúng token là eats; model gán P(eats)=0.70. Tính loss và xem loss thay đổi thế nào khi xác suất đúng bằng 0.95 hoặc 0.01.

In [2]:
import math
import pandas as pd
values = [0.70, 0.95, 0.01]
pd.DataFrame({"P(token đúng)": values,
              "MLM loss -ln(p)": [-math.log(p) for p in values]})

,P(token đúng),MLM loss -ln(p)
0,0.70,0.356675
1,0.95,0.051293
2,0.01,4.605170


Loss nhỏ hơn khi model gán xác suất cao hơn cho token đúng; nếu xác suất đúng rất thấp thì cross-entropy phạt mạnh.

## 10. Calculation Exercise 2 — Tokenization
Tokenize chính xác câu “The cats are playing.” bằng tokenizer pretrained được cung cấp. Không đoán tokenization trước khi gọi tokenizer.

In [3]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
from pretrained_classification import get_tokenizer
tokenizer = get_tokenizer()
text = "The cats are playing."
encoded = tokenizer(text, padding="max_length", truncation=True,
                    max_length=12, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
token_ids = encoded["input_ids"][0].tolist()
mask = encoded["attention_mask"][0].tolist()
pd.DataFrame({"position": range(len(tokens)), "token": tokens,
              "token_id": token_ids, "attention_mask": mask})

D:\NLP_exercise\lab6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,position,token,token_id,attention_mask
0,0,[CLS],101,1
1,1,the,1996,1
2,2,cats,8870,1
3,3,are,2024,1
4,4,playing,2652,1
5,5,.,1012,1
6,6,[SEP],102,1
7,7,[PAD],0,0
8,8,[PAD],0,0
9,9,[PAD],0,0


In [4]:
lexical = [t for t in tokens if t not in tokenizer.all_special_tokens]
summary = pd.DataFrame([{
    "words in raw text": len(text.rstrip(".").split()),
    "tokens excluding special/padding": len(lexical),
    "tokens including [CLS]/[SEP]/padding": len(tokens),
    "special tokens": ", ".join(t for t in tokens if t in tokenizer.all_special_tokens)
}])
summary

,words in raw text,tokens excluding special/padding,tokens including [CLS]/[SEP]/padding,special tokens
0,4,5,12,"[CLS], [SEP], [PAD], [PAD], [PAD], [PAD], [PAD]"


Cột attention_mask có 1 cho token nội dung/special token của chuỗi và 0 cho padding. So sánh số word với token thực tế; WordPiece có thể giữ một từ thành một token hoặc tách nó ra.